## Setup
Run notebooks 01–07 first. Install dependencies once from the project root with `python -m pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"data").is_dir() and (candidate/"notebooks").is_dir(): PROJECT_ROOT=candidate; break
else: raise RuntimeError("Start Jupyter inside the project so its root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"; MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

# 10 · Decision engine and operating trade-offs

A score becomes an action only after policy is applied. This notebook compares approve, review, and decline outcomes, estimates manual-review workload, and exposes cost assumptions. Values are illustrative placeholders and require business evidence before use.

In [ ]:
import joblib
import numpy as np
import pandas as pd

train=pd.read_pickle(INTERMEDIATE_DIR/"train_features.pkl"); artifact=joblib.load(MODEL_DIR/"best_experiment_model.joblib")
features=list(artifact["feature_cols"]); ordered=train.sort_values("TransactionDT",kind="stable").reset_index(drop=True); val=ordered.iloc[int(len(ordered)*0.8):].copy()
X=val[features].replace([np.inf,-np.inf],np.nan)
if artifact.get("imputer") is not None: X=artifact["imputer"].transform(X)
if artifact.get("scaler") is not None: X=artifact["scaler"].transform(X)
prob=artifact["model"].predict_proba(X)[:,1]; y=val["isFraud"].astype(int).to_numpy()
print(f"Validation sample: {len(y):,}; fraud rate: {y.mean():.3%}")

## 1. Set the example operating policy

Low scores are approved, middle scores are reviewed, and high scores are declined. The sample cost proxy treats review as a fixed cost, missed fraud as a loss, and legitimate declines as customer impact. It does not model every downstream outcome.

In [ ]:
APPROVE_BELOW=0.05; DECLINE_AT_OR_ABOVE=0.50
REVIEW_COST=3.00; MISSED_FRAUD_COST=150.00; LEGITIMATE_DECLINE_COST=8.00
MAX_REVIEW_RATE=0.05  # illustrative capacity ceiling
assert 0<=APPROVE_BELOW<DECLINE_AT_OR_ABOVE<=1
def apply_policy(scores,approve_below,decline_at):
    return np.select([scores<approve_below,scores>=decline_at],["approve","decline"],default="review")
action=apply_policy(prob,APPROVE_BELOW,DECLINE_AT_OR_ABOVE)
pd.Series(action).value_counts(normalize=True).rename("share").to_frame()

## 2. Summarize decisions, outcomes, and queue size

In [ ]:
outcomes=pd.DataFrame({"fraud":y,"probability":prob,"action":action})
summary=outcomes.groupby("action").agg(transactions=("fraud","size"),fraud_cases=("fraud","sum"),observed_fraud_rate=("fraud","mean"),mean_score=("probability","mean"))
summary["share"]=summary["transactions"]/len(outcomes); summary

In [ ]:
reviewed=int((action=="review").sum()); missed=int(((action=="approve")&(y==1)).sum()); legit_declined=int(((action=="decline")&(y==0)).sum())
estimated_cost=reviewed*REVIEW_COST+missed*MISSED_FRAUD_COST+legit_declined*LEGITIMATE_DECLINE_COST
review_rate=reviewed/len(y)
print(f"Review queue: {reviewed:,} ({review_rate:.2%}); ceiling: {MAX_REVIEW_RATE:.2%}")
print(f"Missed fraud in approvals: {missed:,}; legitimate declines: {legit_declined:,}")
print(f"Illustrative sample cost: ${estimated_cost:,.2f}")
if review_rate>MAX_REVIEW_RATE: print("Policy exceeds the example review capacity; inspect the threshold grid.")

## 3. Threshold sensitivity under the example assumptions

This grid compares policies and their review rate. Do not select and report a policy on the same validation data as if it were an independent evaluation.

In [ ]:
rows=[]
for approve_below in np.arange(0.01,0.31,0.02):
    for decline_at in np.arange(max(approve_below+0.05,0.20),0.81,0.05):
        a=apply_policy(prob,approve_below,decline_at); nreview=int((a=="review").sum())
        nmiss=int(((a=="approve")&(y==1)).sum()); nbad=int(((a=="decline")&(y==0)).sum())
        cost=nmiss*MISSED_FRAUD_COST+nbad*LEGITIMATE_DECLINE_COST+nreview*REVIEW_COST
        rows.append({"approve_below":approve_below,"decline_at":decline_at,"review_rate":nreview/len(y),"estimated_cost":cost})
grid=pd.DataFrame(rows); feasible=grid[grid.review_rate<=MAX_REVIEW_RATE]
(feasible if not feasible.empty else grid).sort_values("estimated_cost").head(10)

## 4. Operational controls

A real decision service needs analyst guidance, safe fallbacks, customer appeal and override paths, audit logs, model-version tracking, queue prioritization, and monitoring of action rates. Replace every placeholder with approved estimates and evaluate on a later untouched time period.